# Preparación de los datos en Databricks — registro del trabajo realizado

Este notebook no contiene código. Deja constancia de lo que se hizo en Databricks, desde los datos en bruto hasta las ventanas que usa el código de Colab de este repositorio.

**Por qué no se publica el código de esta fase.** Los datos están sujetos a un acuerdo de confidencialidad. El código de Databricks trabaja directamente sobre ellos y contiene nombres de tablas, campos y rutas que no pueden hacerse públicos. Por eso aquí solo se describe cada etapa: qué hace, de qué parte, qué produce y qué decisiones se tomaron.

**Entorno.** Databricks Free Edition (cómputo *serverless*) con PySpark. Los resultados de cada etapa se guardan como tablas Delta.

**Nomenclatura.** Las señales se identifican con nombres genéricos, igual que en la memoria. No se describe qué mide cada una.

| Grupo | Señales |
|---|---|
| Familia S01 | S01ERR, S01REC01, S01REC02 |
| Familia S02 | S02ERR, S02REC01, S02REC02 |
| Familia S03 | S03, S03H01, S03L01 |
| Familia S04 | S04, S04L01 |
| Variables derivadas | V01 (minutos desde el reporte anterior), V02 (indicador 0/1 de salto brusco en S01ERR) |

Los fabricantes y modelos se anonimizan como Fabricante_1, Modelo_1, etc.

## Índice

1. Ingesta
2. Anonimización
3. Exploración de los datos
4. Limpieza
5. Variables derivadas
6. Ventanas y reparto en conjuntos
7. Exportación a Colab
8. Resumen por dispositivo del conjunto de prueba
9. Notebooks auxiliares
10. Comprobaciones posteriores (octubre de 2026)
11. Cifras principales
12. Formato de salida

## 1. Ingesta

- **Qué hace:** carga en Databricks el fichero comprimido con los datos originales y lo guarda como tabla en bruto, sin transformar.
- **Entrada:** fichero de telemetría de un mes, entregado por la empresa.
- **Salida:** tabla en bruto (capa previa a *bronze*), en un esquema privado.
- **Decisiones:**
  - Los contadores se guardan como enteros de 64 bits, para no añadir desbordamientos en la carga.
  - Los campos de fecha y hora se guardan como texto y se convierten más adelante, en la limpieza, para evitar errores de formato en la primera carga.
- **Publicación:** no se publica. Trabaja sobre los datos originales.

*Etapa descrita a alto nivel; el notebook no está en el repositorio.*

## 2. Anonimización

- **Qué hace:** sustituye los identificadores por seudónimos antes de cualquier análisis. Todas las etapas siguientes trabajan solo con datos seudonimizados.
- **Entrada:** tabla en bruto.
- **Salida:** tabla *bronze* seudonimizada.
- **Decisiones:**
  - Identificador del dispositivo y número administrativo: hash SHA-256 con una sal fija y secreta. La sal es fija para que el mismo dispositivo tenga siempre el mismo seudónimo y se puedan construir sus series temporales.
  - URL del canal: hash, conservando solo el tipo de servicio.
  - Dirección IP: eliminada.
  - La sal no se guarda en ningún fichero versionado.
- **Publicación:** no se publica. Contiene la lógica de anonimización.

*Etapa descrita a alto nivel; el notebook no está en el repositorio.*

## 3. Exploración de los datos

- **Qué hace:** estudia los datos antes de transformarlos. No escribe ninguna tabla; solo documenta hallazgos y decide qué hacer con cada uno.
- **Entrada:** tabla *bronze* (18.090.838 reportes de 52.763 dispositivos, 26 campos).

**Calidad de los datos**
- 15 reportes sin identificador de dispositivo.
- Valores ausentes ocultos (texto «NONE») en fabricante (13), modo de trabajo (12) y modelo (9). Son casos independientes, no un lote corrupto.
- 4.247 reportes duplicados, todos con contenido idéntico: reenvíos de la ingesta, no errores de clave.
- Un mismo fabricante aparecía escrito en mayúsculas y en minúsculas.
- Un campo de fecha de carga resultó ser la fecha de apertura de la sesión. Para cualquier análisis temporal se usa la fecha del evento.

**Distribución de los contadores**
- Todos los contadores tienen inflación de ceros y cola muy larga. El método IQR no sirve para detectar extremos: con mediana 0, marca como atípico cualquier valor distinto de cero. Se usan percentiles altos.
- S01ERR tiene valores imposibles: un máximo de 2.128.852.019 y un mínimo de −2.074.940.405, efecto del desbordamiento de un entero de 32 bits.
- S03 acumula el valor 9 en 1.782.398 reportes, entre 10 y 20 veces más que los valores vecinos. Podría ser un tope del sistema; no se ha podido confirmar.
- Un duodécimo contador vale siempre 0 y se descarta.

**Relaciones entre señales**
- S02ERR y S02REC01 están muy correlacionadas (0,97; VIF de 50,8 y 45,4). Se mantienen las dos.
- El análisis de componentes principales necesita 8 de 11 componentes para explicar el 90 % de la varianza: la redundancia está localizada, no es general.

**Comportamiento temporal**
- Ciclo diario de audiencia: de unos 41.000 dispositivos activos de madrugada a unos 51.000 a las 21 h.
- El intervalo entre reportes de un mismo dispositivo forma picos en múltiplos de 30 minutos. Mediana de 30 minutos, percentil 95 de 540 y máximo de unos 29,5 días.
- El 62,5 % de los dispositivos tiene al menos un hueco de más de un día. Las series no son continuas y hay que separarlas en sesiones.

**Cambio de régimen el 13/11**
- Desde ese día el número de reportes cae alrededor de un 50 % y no se recupera hasta el final del periodo.
- El número de dispositivos activos se mantiene (unos 44.000–47.000). Lo que baja es el número de reportes por dispositivo.
- La mediana de S01ERR y de S03 pasa de 1 a 0.
- La cadencia de reporte (unos 30 minutos) no cambia. El dispositivo solo informa cuando algún contador es distinto de cero, así que menos reportes producen huecos más largos.
- Con estos datos no se puede saber si la causa es un cambio en la captura o una caída real de los errores.

- **Decisiones que salen de aquí:** limpieza de la etapa 4, variables derivadas de la etapa 5, ventanas por sesión y reparto por régimen de la etapa 6.
- **Memoria:** apartados 4.2.1 y 4.2.3.

## 4. Limpieza

- **Qué hace:** aplica las decisiones de la exploración.
- **Entrada:** tabla *bronze*.
- **Salida:** tabla *silver*.
- **Pasos:**
  1. Se eliminan los 15 reportes sin identificador de dispositivo. No se pueden asignar a ninguna serie.
  2. Se rellenan fabricante, modo de trabajo y modelo con el valor más frecuente del propio dispositivo en sus otros reportes. Se recuperan todos los casos (13, 7 y 5).
  3. Se eliminan los 4.247 duplicados exactos, conservando una copia de cada reporte.
  4. Se descartan el contador siempre a 0 y el modo de trabajo, que tras el relleno resulta constante.
  5. Se unifica el fabricante en minúsculas.
  6. Se convierten los campos de fecha y hora de texto a marca de tiempo.
- **Memoria:** apartado 4.2.1.

## 5. Variables derivadas

- **Qué hace:** añade las variables que no existen en los datos originales.
- **Entrada:** tabla *silver*.
- **Salida:** tabla de variables (18.090.838 filas).
- **Variables:**
  - **Tipo de servicio** (tres categorías) e **identificador de canal**, a partir de la URL seudonimizada. Solo sirven para describir y estratificar; no entran en el modelo.
  - **V02:** vale 1 si S01ERR cambia en más de 10⁶ respecto al reporte anterior del mismo dispositivo. La primera lectura de cada dispositivo vale 0.
  - **Número de saltos por dispositivo:** total de reportes con V02 = 1 en todo el mes. Solo se usa como metadato, nunca como entrada, porque repite en todas las filas un valor calculado con el futuro del dispositivo (fuga de información).
  - **V01:** minutos desde el reporte anterior del mismo dispositivo.
- **Análisis del desbordamiento de S01ERR:**
  - Los valores negativos proceden de un único dispositivo de Fabricante_1, dentro de una sesión de unas 21 horas.
  - Los saltos de más de 10⁶ aparecen todos los días en unos pocos dispositivos (5–30 al día), con un pico el 24/11 (927 dispositivos), repartido entre los cuatro fabricantes.
  - En esos reportes S03 está algo más alto de lo habitual. No se puede descartar que haya también una degradación real.
  - En esta etapa no se corrige ni se elimina nada; solo se marca con V02. La exclusión de las ventanas afectadas se hace en Colab.
- **Memoria:** apartado 4.2.1 y Tabla 6.

## 6. Ventanas y reparto en conjuntos

- **Qué hace:** construye las secuencias de entrada de TranAD y TranAD+ y las reparte en entrenamiento, validación y prueba.
- **Entrada:** tabla de variables.
- **Salida:** tres tablas de ventanas (entrenamiento, validación y prueba).

**Construcción de las ventanas**
- 12 reportes consecutivos del mismo dispositivo, sin solapamiento. Doce reportes cubren unas 6 horas con la cadencia de 30 minutos.
- La secuencia se corta cuando V01 > 150 minutos (cinco intervalos de reporte). Así una ventana no une reportes muy separados.
- Si al final de una sesión quedan menos de 12 reportes, se descartan. No se rellena con valores artificiales.
- Se construyen sobre todo el periodo (27/10–28/11), sin excluir ningún tramo.

**Variables de cada reporte (13, en este orden)**

S01ERR, S01REC01, S01REC02, S03, S04, S04L01, S03H01, S03L01, S02ERR, S02REC01, S02REC02, V01, V02

- V02 se pasa a entero (0/1). V01 vale 0 en el primer reporte de cada dispositivo, que no tiene reporte anterior.
- Fabricante, modelo y tipo de servicio no entran en el modelo, para que no aprenda atajos como «este fabricante, esta anomalía».

**Reparto en conjuntos**
- Reparto cronológico 70/15/15, **por separado dentro de cada régimen** (antes y después del 13/11), que luego se une.
- Motivo: un corte cronológico único dejaría todo el entrenamiento en el primer régimen y toda la prueba en el segundo.
- Los cortes se calculan por días completos, según el volumen acumulado de ventanas de cada día. Un primer intento por número de días daba un reparto de 78/13/9.
- Dentro de cada régimen, las ventanas de entrenamiento son anteriores a las de validación, y estas a las de prueba.

| Conjunto | Ventanas |
|---|---|
| Entrenamiento | 563.598 |
| Validación | 113.125 |
| Prueba | 88.197 |
| **Total** | **764.920** |

- **Memoria:** apartados 4.2.2 y 4.2.3, Tabla 5.

## 7. Exportación a Colab

- **Qué hace:** exporta las tres tablas de ventanas a ficheros Parquet para usarlas en Colab.
- **Entrada:** tablas de ventanas.
- **Salida:** carpeta con tres subcarpetas (entrenamiento, validación y prueba), con 5, 1 y 1 ficheros respectivamente.
- **Columnas exportadas:** identificador seudonimizado del dispositivo, sesión, número de ventana, secuencia de 12 × 13 valores, inicio y fin de la ventana.
- Los ficheros se descargan a mano desde Databricks y se suben a Google Drive. A partir de aquí todo el trabajo se hace en Colab.

## 8. Resumen por dispositivo del conjunto de prueba

- **Qué hace:** calcula, para cada dispositivo del conjunto de prueba, la media, la desviación típica y el máximo de los 11 contadores en todo el mes, más su número total de reportes.
- **Entrada:** tabla de variables y tabla de ventanas de prueba.
- **Salida:** un fichero CSV.
- **Uso:** dar al modelo de lenguaje el comportamiento habitual del dispositivo en pruebas exploratorias de Colab. No interviene en los experimentos finales de la memoria.

## 9. Notebooks auxiliares

- **Visualizaciones de los datos:** gráficas de apoyo durante la exploración (evolución diaria, huecos entre reportes, series de dispositivos concretos, efecto de la deduplicación y de la segmentación en sesiones).
- **Extracción de ventanas en bruto:** recupera de la tabla de variables los valores originales de ventanas concretas, para revisarlas a mano o para los experimentos con el modelo de lenguaje.

Ninguno de los dos modifica las tablas del proceso.

## 10. Comprobaciones posteriores (octubre de 2026)

Se hicieron después de entrenar los detectores, para revisar si los datos de entrenamiento contenían comportamientos que debieron excluirse.

### 10.1 Corte en la captura de datos (2/11–3/11)

- **Qué se observó:** entre el 2/11 hacia las 22:30 y el 3/11 hacia las 07:30, unos 21.000 dispositivos dejan de reportar de forma escalonada y vuelven todos a la vez. Es el único corte de esa magnitud en el mes.
- **Duda:** si el primer reporte tras el corte acumulaba los errores de esas horas, el entrenamiento contendría valores inflados.
- **Comprobación:** se comparó el primer reporte de las 07:00–07:59 del 3/11 tras una pausa de al menos 400 minutos (19.107 dispositivos) con el de la misma franja de los demás días tras una pausa parecida. Se suman los 11 contadores y se excluyen los reportes con desbordamiento.

| Grupo | Mediana | Percentil 90 | Spearman (pausa frente a total) |
|---|---|---|---|
| Mañana del 3/11 | 3 | 41 | 0,075 |
| Otras mañanas | 4 | 51 | −0,111 |

- **Conclusión:** no hay acumulación. El primer reporte tras el corte es normal y no crece con la duración de la pausa. El corte no se filtra. Su único efecto era el hueco de tiempo en V01, y la segmentación en sesiones ya impide que una ventana una reportes de antes y de después.

### 10.2 Meseta de S04L01

- **Qué se observó:** S04L01 se queda fijo en un mismo valor durante casi toda la ventana, sin actividad en ningún otro contador (apartado 4.2.1 de la memoria).
- **Niveles encontrados:**
  - 1384–1387 y 2770: el nivel ya descrito en la memoria.
  - 23: misma huella. En el 98 % de sus ventanas S04L01 es el único contador activo, frente al 98,7 % del nivel conocido. 8 dispositivos presentan los dos niveles y reúnen el 77 % de las ventanas con meseta en 23.
  - 1: no es meseta. Nunca aparece solo; va acompañado de errores en S01ERR (93 % de las ventanas), S03 (96 %) y S04 (90 %). Es un comportamiento habitual que acompaña a otros errores.
- **Criterio para identificar las ventanas con meseta:**
  - 1384–1387 o 2770 en algún reporte de la ventana. Estos valores solo aparecen ligados al fallo: en las ventanas que los contienen junto a otros contadores activos, la mayoría son bordes de una meseta o mesetas cortas.
  - 23 en algún reporte **y** S04L01 como único contador activo de la ventana. El 23 también aparece en ventanas normales, así que hace falta la segunda condición.
- **Evolución temporal:** hasta el 7/11 hay alrededor de 100 dispositivos al día con meseta. El 8/11 el número sube de golpe a unos 700 y después baja poco a poco hasta unos 200 a final de mes. Que tantos dispositivos empiecen el mismo día apunta a una causa común del sistema.
- **Estado:** no se reentrenó. La meseta se recoge como limitación en la memoria (apartados 4.2.1, 5.2.5 y 6.3).
- **Nota:** estas comprobaciones se calcularon sobre la primera versión de las tablas de ventanas, anterior al reparto por régimen. Los criterios y la evolución diaria no dependen del reparto, pero los recuentos por conjunto deben recalcularse con las tablas finales antes de citarlos.

## 11. Cifras principales

| Concepto | Valor |
|---|---|
| Reportes en bruto | 18.090.838 |
| Dispositivos | 52.763 |
| Periodo | 27/10/2017 – 28/11/2017 |
| Reportes sin identificador eliminados | 15 |
| Duplicados eliminados | 4.247 |
| Ventanas construidas | 764.920 |
| Ventanas con valores imposibles (excluidas en Colab) | 885 |
| Ventanas tras la exclusión: entrenamiento / validación / prueba | 562.875 / 113.034 / 88.126 |

## 12. Formato de salida

Es el formato que espera el código de Colab de este repositorio. Quien quiera ejecutarlo con sus propios datos debe preparar ficheros con esta estructura.

- **Ficheros:** Parquet, uno o varios por conjunto, en tres carpetas: `train`, `val` y `test`.
- **Una fila por ventana**, con estas columnas:

| Columna | Tipo | Contenido |
|---|---|---|
| `device_pseudo_id` | texto | Identificador seudonimizado del dispositivo |
| `session_id` | entero | Número de sesión dentro del dispositivo |
| `window_id` | entero | Número de ventana dentro de la sesión |
| `secuencia` | lista de 12 listas de 13 números | Los 12 reportes de la ventana, cada uno con las 13 variables en el orden de la etapa 6 |
| `inicio_ventana` | marca de tiempo | Hora del primer reporte |
| `fin_ventana` | marca de tiempo | Hora del último reporte |

- Los valores de `secuencia` van sin escalar. El escalado (logaritmo y mínimo-máximo ajustado solo con entrenamiento) y la exclusión de valores imposibles se hacen en Colab.